# 7 · Classification — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch07_classification/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch07_classification/03_notebook.ipynb`, créée par `python tools/start_chapter.py 7`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 7.Q1 à 7.Q11, 7.R1 à 7.R3, 7.1 à 7.6 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ | ★ à ★★★ | |
| A | 7.11 à 7.16 | frontières, distances, centroïde le plus proche, carte de probabilité, un-contre-tous avec des centroïdes (*prochaine session de génération*) | | | |
| B | 7.17 à 7.21 | k-means, DBSCAN et HDBSCAN avec scikit-learn, puis la grande dimension (*prochaine session de génération*) | | | |
| C | 7.22 à 7.24 | un-contre-tous et un-contre-un génériques dans mylearn.multiclass (*prochaine session de génération*) | | | |
| D | 7.25 à 7.31 | k-means de zéro dans mylearn.cluster, silhouette, choix de k, phénomène de Hughes, défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")  # your own library

## Objectifs et rappel express

- Lire des régions et des frontières de décision, et régler un seuil d'après le coût des erreurs.
- Traiter plusieurs classes avec des classifieurs binaires : un-contre-tous et un-contre-un.
- Programmer le centroïde le plus proche et k-means (Lloyd, k-means++), et choisir $k$ avec l'inertie et la silhouette ; savoir quand préférer DBSCAN ou HDBSCAN.
- Mesurer ce que la dimension fait aux densités et aux distances.

**Rappel express.** $N_{\text{OvR}} = K$, $N_{\text{OvO}} = \frac{K(K-1)}{2}$ ; seuil de coût minimal $t^* = \frac{C_{FP}}{C_{FP} + C_{FN}}$ ; $\lVert \mathbf{a} - \mathbf{b} \rVert^2 = \lVert \mathbf{a} \rVert^2 - 2\,\mathbf{a} \cdot \mathbf{b} + \lVert \mathbf{b} \rVert^2$ ; inertie $J = \sum_i \lVert \mathbf{x}_i - \boldsymbol{\mu}_{c_i} \rVert^2$ ; silhouette $s = \frac{b - a}{\max(a, b)}$ ; densité $\rho = \frac{n}{b^d}$ ; peau d'une boule $1 - (1 - \varepsilon)^d$. En Python : `sklearn.cluster.KMeans`, `DBSCAN`, `HDBSCAN`, `sklearn.metrics.silhouette_score`, `sklearn.multiclass`, `rng.choice(n, p=w / w.sum())`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 7.1 à 7.4, ∂ 7.5 et 7.6)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.375` (en Python, le séparateur décimal est un **point** ; `0,375` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"D"`) ; plusieurs nombres : une liste (`[2, 5]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », le quiz Q10, ∂ 7.7, la réflexion et l'entretien se corrigent avec `05_solutions.md`.

**7.Q1 — Label, prédiction, vérité terrain : le vocabulaire**

In [ ]:
# 7.Q1: the values you found on paper
answer_7_Q1a = ...  # a) the letter of the expression that is NOT the expert's class, e.g. "A"
answer_7_Q1b = ...  # b) True or False
answer_7_Q1c = ...  # c) True or False

for letter, answer in zip("abc", [answer_7_Q1a, answer_7_Q1b, answer_7_Q1c]):
    wb.check(f"7.Q1{letter}", answer)

**7.Q2 — Binaire, multi-classe ou multi-étiquette ? Cinq situations**

In [ ]:
# 7.Q2: the values you found on paper
answer_7_Q2a = ...  # a) "B", "C" or "E"
answer_7_Q2b = ...  # b) "B", "C" or "E"
answer_7_Q2c = ...  # c) "B", "C" or "E"
answer_7_Q2d = ...  # d) "B", "C" or "E"
answer_7_Q2e = ...  # e) "B", "C" or "E"

for letter, answer in zip("abcde", [answer_7_Q2a, answer_7_Q2b, answer_7_Q2c, answer_7_Q2d, answer_7_Q2e]):
    wb.check(f"7.Q2{letter}", answer)

**7.Q3 — Régions et frontières de décision**

In [ ]:
# 7.Q3: the values you found on paper
answer_7_Q3a = ...  # a) True or False
answer_7_Q3b = ...  # b) a whole number
answer_7_Q3c = ...  # c) True or False
answer_7_Q3d = ...  # d) True or False
answer_7_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_7_Q3a, answer_7_Q3b, answer_7_Q3c, answer_7_Q3d, answer_7_Q3e]):
    wb.check(f"7.Q3{letter}", answer)

**7.Q4 — Classes qui se recouvrent : probabilités et politique de seuil**

In [ ]:
# 7.Q4: the values you found on paper
answer_7_Q4a = ...  # a) True or False
answer_7_Q4b = ...  # b) True or False
answer_7_Q4c = ...  # c) "A", "B" or "C"
answer_7_Q4d = ...  # d) the threshold, 3 decimals

for letter, answer in zip("abcd", [answer_7_Q4a, answer_7_Q4b, answer_7_Q4c, answer_7_Q4d]):
    wb.check(f"7.Q4{letter}", answer)

**7.Q5 — Cinq mesures par œuf : ce que change (ou non) la dimension**

In [ ]:
# 7.Q5: the values you found on paper
answer_7_Q5a = ...  # a) a whole number
answer_7_Q5b = ...  # b) True or False
answer_7_Q5c = ...  # c) True or False
answer_7_Q5d = ...  # d) a whole number

for letter, answer in zip("abcd", [answer_7_Q5a, answer_7_Q5b, answer_7_Q5c, answer_7_Q5d]):
    wb.check(f"7.Q5{letter}", answer)

**7.Q6 — Un-contre-tous : combien de modèles, quelle décision ?**

In [ ]:
# 7.Q6: the values you found on paper
answer_7_Q6a = ...  # a) a whole number
answer_7_Q6b = ...  # b) a letter
answer_7_Q6c = ...  # c) True or False
answer_7_Q6d = ...  # d) a letter

for letter, answer in zip("abcd", [answer_7_Q6a, answer_7_Q6b, answer_7_Q6c, answer_7_Q6d]):
    wb.check(f"7.Q6{letter}", answer)

**7.Q7 — Un-contre-un : duels, votes et coût**

In [ ]:
# 7.Q7: the values you found on paper
answer_7_Q7a = ...  # a) a whole number
answer_7_Q7b = ...  # b) True or False
answer_7_Q7c = ...  # c) True or False
answer_7_Q7d = ...  # d) a letter
answer_7_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_7_Q7a, answer_7_Q7b, answer_7_Q7c, answer_7_Q7d, answer_7_Q7e]):
    wb.check(f"7.Q7{letter}", answer)

**7.Q8 — Ce que k-means ne peut pas deviner tout seul**

In [ ]:
# 7.Q8: the values you found on paper
answer_7_Q8a = ...  # a) True or False
answer_7_Q8b = ...  # b) a letter
answer_7_Q8c = ...  # c) True or False
answer_7_Q8d = ...  # d) True or False
answer_7_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_7_Q8a, answer_7_Q8b, answer_7_Q8c, answer_7_Q8d, answer_7_Q8e]):
    wb.check(f"7.Q8{letter}", answer)

**7.Q9 — Densité d'échantillons quand les features s'accumulent**

In [ ]:
# 7.Q9: the values you found on paper
answer_7_Q9a = ...  # a) a whole number
answer_7_Q9b = ...  # b) 2 decimals
answer_7_Q9c = ...  # c) 4 decimals
answer_7_Q9d = ...  # d) True or False
answer_7_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_7_Q9a, answer_7_Q9b, answer_7_Q9c, answer_7_Q9d, answer_7_Q9e]):
    wb.check(f"7.Q9{letter}", answer)

**7.Q11 — Géométrie déroutante en grande dimension**

In [ ]:
# 7.Q11: the values you found on paper
answer_7_Q11a = ...  # a) True or False
answer_7_Q11b = ...  # b) True or False
answer_7_Q11c = ...  # c) True or False
answer_7_Q11d = ...  # d) a letter

for letter, answer in zip("abcd", [answer_7_Q11a, answer_7_Q11b, answer_7_Q11c, answer_7_Q11d]):
    wb.check(f"7.Q11{letter}", answer)

**7.R1 — Ch. 6 : entropie d'un cluster pur et d'un cluster mélangé**

In [ ]:
# 7.R1: the values you found on paper
answer_7_R1a = ...  # a) in bits
answer_7_R1b = ...  # b) in bits
answer_7_R1c = ...  # c) in bits
answer_7_R1d = ...  # d) in bits, 3 decimals
answer_7_R1e = ...  # e) in bits, 3 decimals

for letter, answer in zip("abcde", [answer_7_R1a, answer_7_R1b, answer_7_R1c, answer_7_R1d, answer_7_R1e]):
    wb.check(f"7.R1{letter}", answer)

**7.R2 — Ch. 4 : probabilité a posteriori « fécondé » par la règle de Bayes**

In [ ]:
# 7.R2: the values you found on paper
answer_7_R2a = ...  # a) 3 decimals
answer_7_R2b = ...  # b) True or False
answer_7_R2c = ...  # c) 3 decimals
answer_7_R2d = ...  # d) 3 decimals

for letter, answer in zip("abcd", [answer_7_R2a, answer_7_R2b, answer_7_R2c, answer_7_R2d]):
    wb.check(f"7.R2{letter}", answer)

**7.R3 — 0B : développer ‖a − b‖² avec le produit scalaire**

In [ ]:
# 7.R3: the values you found on paper
answer_7_R3a = ...  # a) a whole number
answer_7_R3b = ...  # b) a whole number
answer_7_R3c = ...  # c) a whole number
answer_7_R3d = ...  # d) a whole number

for letter, answer in zip("abcd", [answer_7_R3a, answer_7_R3b, answer_7_R3c, answer_7_R3d]):
    wb.check(f"7.R3{letter}", answer)

**Ex 7.1 — Compter les classifieurs OvR et OvO**

In [ ]:
# Ex 7.1: the values you found on paper
answer_7_1a = ...  # a) [N_OvR, N_OvO] for K = 3
answer_7_1b = ...  # b) [N_OvR, N_OvO] for K = 10
answer_7_1c = ...  # c) [N_OvR, N_OvO] for K = 26
answer_7_1d = ...  # d) [N_OvR, N_OvO] for K = 100
answer_7_1e = ...  # e) the smallest K
answer_7_1f = ...  # f) total number of training examples, one-versus-rest
answer_7_1g = ...  # g) total number of training examples, one-versus-one
answer_7_1h = ...  # h) how many times cheaper, 1 decimal

for letter, answer in zip("abcdefgh", [answer_7_1a, answer_7_1b, answer_7_1c, answer_7_1d, answer_7_1e, answer_7_1f, answer_7_1g, answer_7_1h]):
    wb.check(f"7.1{letter}", answer)

**Ex 7.2 — Dépouiller les votes d'un un-contre-un à quatre classes**

In [ ]:
# Ex 7.2: the values you found on paper
answer_7_2a = ...  # a) votes [A, B, C, D], first point
answer_7_2b = ...  # b) a letter
answer_7_2c = ...  # c) a whole number
answer_7_2d = ...  # d) votes [A, B, C, D], second point
answer_7_2e = ...  # e) a letter
answer_7_2f = ...  # f) a letter
answer_7_2g = ...  # g) True or False
answer_7_2h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_7_2a, answer_7_2b, answer_7_2c, answer_7_2d, answer_7_2e, answer_7_2f, answer_7_2g, answer_7_2h]):
    wb.check(f"7.2{letter}", answer)

**Ex 7.3 — Une itération de k-means à la main**

In [ ]:
# Ex 7.3: the values you found on paper
answer_7_3a = ...  # a) cluster (1 or 2) of P1 to P6
answer_7_3b = ...  # b) the inertia with the starting centres
answer_7_3c = ...  # c) the two new centres [[x1, y1], [x2, y2]], 2 decimals
answer_7_3d = ...  # d) 2 decimals
answer_7_3e = ...  # e) the number of the point (1 to 6)
answer_7_3f = ...  # f) the centres after the second update, 3 decimals
answer_7_3g = ...  # g) the final inertia, 2 decimals
answer_7_3h = ...  # h) True or False
answer_7_3i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_7_3a, answer_7_3b, answer_7_3c, answer_7_3d, answer_7_3e, answer_7_3f, answer_7_3g, answer_7_3h, answer_7_3i]):
    wb.check(f"7.3{letter}", answer)

**Ex 7.4 — Densité d'échantillons et nombre d'œufs nécessaires**

In [ ]:
# Ex 7.4: the values you found on paper
answer_7_4a = ...  # a) [d = 1, d = 2, d = 3, d = 4], 3 decimals
answer_7_4b = ...  # b) a whole number
answer_7_4c = ...  # c) a whole number
answer_7_4d = ...  # d) a whole number
answer_7_4e = ...  # e) the smallest dimension
answer_7_4f = ...  # f) 3 decimals
answer_7_4g = ...  # g) 2 decimals

for letter, answer in zip("abcdefg", [answer_7_4a, answer_7_4b, answer_7_4c, answer_7_4d, answer_7_4e, answer_7_4f, answer_7_4g]):
    wb.check(f"7.4{letter}", answer)

**Ex 7.5 — Le rayon de l'hyper-orange : r(d) = √d − 1**

In [ ]:
# Ex 7.5: the values you found on paper
answer_7_5a = ...  # a) r(2), 3 decimals
answer_7_5b = ...  # b) r(3), 3 decimals
answer_7_5c = ...  # c) a whole number
answer_7_5d = ...  # d) the smallest dimension
answer_7_5e = ...  # e) the smallest dimension
answer_7_5f = ...  # f) r(50), 3 decimals

for letter, answer in zip("abcdef", [answer_7_5a, answer_7_5b, answer_7_5c, answer_7_5d, answer_7_5e, answer_7_5f]):
    wb.check(f"7.5{letter}", answer)

**Ex 7.6 — Boule dans un cube : rapport des volumes par récurrence**

In [ ]:
# Ex 7.6: the values you found on paper
answer_7_6a = ...  # a) V4, 3 decimals
answer_7_6b = ...  # b) V5, 3 decimals
answer_7_6c = ...  # c) q4, 3 decimals
answer_7_6d = ...  # d) q10 in percent, 3 decimals
answer_7_6e = ...  # e) a dimension
answer_7_6f = ...  # f) 3 decimals

for letter, answer in zip("abcdef", [answer_7_6a, answer_7_6b, answer_7_6c, answer_7_6d, answer_7_6e, answer_7_6f]):
    wb.check(f"7.6{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu choisir un seuil de décision d'après le coût des deux erreurs, et dire qui doit fixer ces coûts ?
2. Sais-tu compter et dépouiller les votes d'un un-contre-un, et dire quand le préférer à l'un-contre-tous ?
3. Sais-tu dérouler k-means à la main, et expliquer pourquoi la densité des exemples s'effondre quand on ajoute des features ?

**Pour aller plus loin** : les guides « Clustering » et « Multiclass and multioutput algorithms » de scikit-learn, cités dans la fiche. La suite : le ch. 8 (entraînement et test), où un jeu de validation fixe les seuils et les hyperparamètres comme $k$.

*Les exercices de code 7.11 à 7.31 (parties A à D, tes librairies `mylearn.cluster` et `mylearn.multiclass`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 7` pour obtenir le notebook complet.*